# DMAS results

Reads `results/results.csv` (one row per question per seed across every
experiment ever run). When the same configuration was run with multiple
seeds, those rows are AVERAGED together per (question, configuration) so
every metric is the seed-mean (with stdev shown alongside).

In [ ]:
import re
from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from IPython.display import display, Markdown

RESULTS_CSV = Path('results/results.csv')
df = pd.read_csv(RESULTS_CSV)

# experiment_id is a 12-char hash of the configuration columns (dedupe key
# minus seed + question), written by experiments/experiment.py. All seed×question
# rows of the same configuration share an experiment_id, so we group on it
# instead of parsing experiment_name.

print(f'rows={len(df)}  experiments={df["experiment_name"].nunique()}  '
      f'configurations={df["experiment_id"].nunique()}')
df.head()

## Seed-averaged dataframe

For each unique `(experiment_id, conversation_index, question)` we collapse
all seed rows into one row that carries:
- the **mean** of every numeric metric
- the **stdev** of f1 / judge_label / api_latency / openai_cost (the columns where variance matters)
- the seed count `n_seeds`

If a configuration was run only once, the stdev columns are `0.0`.

In [ ]:
NUMERIC_COLS = [
    'f1', 'string_sim', 'experiment_duration_s',
    'agent1_tokens','agent2_tokens','agent3_tokens','agent1_cost_usd','agent2_cost_usd','agent3_cost_usd','agent1_memory_tokens','agent2_memory_tokens','agent3_memory_tokens','agent1_memory_cost_usd','agent2_memory_cost_usd','agent3_memory_cost_usd',
    'cpu_edge_ns', 'cpu_cloud_ns',
    'ram_edge_bytes', 'ram_cloud_bytes',
    'disk_edge_bytes', 'disk_cloud_bytes',
    'network_edge_bytes', 'network_cloud_bytes',
    'measured_latency_ms', 'peer_memories', 'own_memories',
    'toxic_latency', 'toxic_jitter', 'toxic_bandwidth', 'peer_threshold_ms',
]
STD_COLS = ['f1', 'agent1_cost_usd','agent2_cost_usd','agent3_cost_usd', 'judge_label']
GROUP_COLS = ['experiment_id', 'memory', 'dataset', 'conversation_index', 'question', 'category', 'category_label', 'gold_answer']

_present_numeric = [c for c in NUMERIC_COLS if c in df.columns]
_present_group   = [c for c in GROUP_COLS  if c in df.columns]

# Coerce judge_label to 0/1 for averaging
if 'judge_label' in df.columns:
    df['judge_label_num'] = df['judge_label'].map({True: 1.0, False: 0.0, 'True': 1.0, 'False': 0.0})

# Coerce peers_asked similarly
if 'peers_asked' in df.columns:
    df['peers_asked_num'] = df['peers_asked'].map({True: 1.0, False: 0.0, 'True': 1.0, 'False': 0.0})

_mean_cols = _present_numeric + (['judge_label_num'] if 'judge_label_num' in df.columns else []) \
                              + (['peers_asked_num'] if 'peers_asked_num' in df.columns else [])
_std_cols  = [c for c in STD_COLS if c in df.columns or c == 'judge_label']

def _seed_mean(group: pd.DataFrame) -> pd.Series:
    out = {}
    out['n_seeds'] = group['seed'].nunique()
    for c in _mean_cols:
        out[c] = pd.to_numeric(group[c], errors='coerce').mean()
    # stdev columns with explicit suffix
    for c in _std_cols:
        col = 'judge_label_num' if c == 'judge_label' else c
        if col in group.columns:
            v = pd.to_numeric(group[col], errors='coerce').dropna()
            out[f'{c}_std'] = float(v.std(ddof=1)) if len(v) >= 2 else 0.0
    # keep first non-null answer for inspection
    answers = group['answer'].dropna().astype(str)
    out['answer_first_seed'] = answers.iloc[0] if len(answers) else ''
    return pd.Series(out)

agg = (df.groupby(_present_group, dropna=False)
         .apply(_seed_mean, include_groups=False)
         .reset_index())

# Friendly rename: judge_label_num → judge_acc, peers_asked_num → peers_asked_rate
agg = agg.rename(columns={'judge_label_num': 'judge_acc',
                          'peers_asked_num': 'peers_asked_rate',
                          'judge_label_std': 'judge_acc_std'})

print(f'seed-averaged rows: {len(agg)} (vs {len(df)} raw)')
agg.head()

## Top-level overview (per configuration, seed-averaged)

In [ ]:
overview = (
    agg.groupby('experiment_id')
       .agg(n_questions=('question', 'count'),
            seeds=('n_seeds', 'max'),
            f1_mean=('f1', 'mean'),
            f1_std_mean=('f1_std', 'mean'),
            judge_acc=('judge_acc', 'mean'),
            cost_total=('agent1_cost_usd','agent2_cost_usd','agent3_cost_usd', 'sum'),
            latency_p50=(lambda s: s.quantile(0.50)),
            latency_p95=(lambda s: s.quantile(0.95)))
       .sort_values('f1_mean', ascending=False)
)
overview

## Per-configuration detail

In [ ]:
NUMERIC_PLOT_COLS = [
    'f1',
    'agent1_tokens','agent2_tokens','agent3_tokens','agent1_cost_usd','agent2_cost_usd','agent3_cost_usd','agent1_memory_tokens','agent2_memory_tokens','agent3_memory_tokens','agent1_memory_cost_usd','agent2_memory_cost_usd','agent3_memory_cost_usd',
    'cpu_edge_ns', 'cpu_cloud_ns',
    'ram_edge_bytes', 'ram_cloud_bytes',
    'disk_edge_bytes', 'disk_cloud_bytes',
    'network_edge_bytes', 'network_cloud_bytes',
    'measured_latency_ms',
]


def _summary_table(g):
    rows = []
    rows.append(('n_questions', len(g)))
    rows.append(('seeds', int(g['n_seeds'].max()) if len(g) else 0))
    rows.append(('f1_mean', g['f1'].mean()))
    rows.append(('f1_stdev (across seeds, avg per question)', g.get('f1_std', pd.Series([0])).mean()))
    rows.append(('judge_acc', g['judge_acc'].mean() if 'judge_acc' in g.columns else float('nan')))
    rows.append(('tokens_total', g[['agent1_tokens','agent2_tokens','agent3_tokens']].sum().sum() if 'openai_tokens' in g.columns else 0))
    rows.append(('cost_total_usd', g['agent1_cost_usd'].sum(min_count=1) if 'agent1_cost_usd' in g.columns else 0))
    rows.append(('latency_s_p50', g[].quantile(0.50)))
    rows.append(('latency_s_p95', g[].quantile(0.95)))
    rows.append(('latency_s_p99', g[].quantile(0.99)))
    if 'peers_asked_rate' in g.columns:
        rows.append(('peers_asked_rate', g['peers_asked_rate'].mean()))
    return pd.DataFrame(rows, columns=['metric', 'value']).set_index('metric')


def _category_table(g):
    col = None
    if 'category_label' in g.columns and g['category_label'].notna().any():
        col = 'category_label'
    elif 'category' in g.columns and g['category'].notna().any():
        col = 'category'
    if col is None:
        return None
    aggs = {'n': ('question', 'count'), 'f1_mean': ('f1', 'mean')}
    if 'judge_acc' in g.columns:
        aggs['judge_acc'] = ('judge_acc', 'mean')
    return (g.groupby(col).agg(**aggs).sort_values('f1_mean', ascending=False))


def _plot_category_bar(cat, title):
    if cat is None or cat.empty:
        return
    cols = [c for c in ['f1_mean', 'judge_acc'] if c in cat.columns]
    fig, ax = plt.subplots(figsize=(8, 3))
    cat[cols].plot.bar(ax=ax, alpha=0.85)
    ax.set_title(f'{title} -- per category')
    ax.set_ylim(0, 1)
    ax.set_ylabel('score')
    ax.legend(loc='upper right', fontsize=8)
    ax.tick_params(axis='x', rotation=30)
    fig.tight_layout()
    plt.show()


def _plot_distributions(g, title):
    cols = [c for c in NUMERIC_PLOT_COLS if c in g.columns and g[c].notna().any()]
    if not cols:
        return
    n = len(cols)
    rows_n = (n + 2) // 3
    fig, axes = plt.subplots(rows_n, 3, figsize=(12, 3 * rows_n))
    axes = axes.flatten() if hasattr(axes, 'flatten') else [axes]
    for ax in axes[n:]:
        ax.axis('off')
    for ax, c in zip(axes, cols):
        v = pd.to_numeric(g[c], errors='coerce').dropna()
        if v.empty:
            ax.set_visible(False)
            continue
        ax.hist(v, bins=30, alpha=0.85, edgecolor='black', linewidth=0.3)
        ax.set_title(c, fontsize=10)
        ax.tick_params(labelsize=8)
    fig.suptitle(title, fontsize=12)
    fig.tight_layout()
    plt.show()

In [ ]:
for name, g in agg.groupby('experiment_id'):
    n_seeds = int(g['n_seeds'].max()) if len(g) else 0
    suffix = f' ({n_seeds} seeds, averaged)' if n_seeds >= 2 else ' (single seed)'
    display(Markdown(f'### `{name}`{suffix}'))
    display(_summary_table(g))
    cat = _category_table(g)
    if cat is not None:
        display(cat)
        _plot_category_bar(cat, name)
    _plot_distributions(g, name)

## Cross-configuration comparisons

Box plots over the seed-averaged values (one observation per question per
configuration), so configurations with multiple seeds aren't double-counted.

In [ ]:
fig, ax = plt.subplots(figsize=(10, 4))
agg.boxplot(column=by='experiment_id', ax=ax, rot=30)
ax.set_title('API latency (s) by configuration (seed-averaged)')
ax.set_ylabel('seconds')
plt.suptitle('')
plt.tight_layout()
plt.show()

fig, ax = plt.subplots(figsize=(10, 4))
agg.boxplot(column='f1', by='experiment_id', ax=ax, rot=30)
ax.set_title('F1 by configuration (seed-averaged)')
ax.set_ylim(0, 1)
plt.suptitle('')
plt.tight_layout()
plt.show()